# Text

RuSpellGold: 1711 пар «исходный текст — исправление»


## Библиотеки

Устанавливаем недостающие зависимости из `pyproject.toml`


In [1]:
from pathlib import Path
from importlib import invalidate_caches, metadata
import subprocess
import sys
import tomllib

if sys.version_info < (3, 11):
    raise RuntimeError('Для проекта нужен Python 3.11 или новее.')

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Откройте ноутбук внутри папки проекта с pyproject.toml.')
config = tomllib.loads((ROOT / 'pyproject.toml').read_text(encoding='utf-8'))

# Проверяем версии библиотек.
try:
    metadata.version('packaging')
except metadata.PackageNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'packaging>=23'])
    invalidate_caches()
from packaging.requirements import Requirement

to_install = []
to_update = []
for specification in config['project']['dependencies']:
    requirement = Requirement(specification)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    try:
        installed = metadata.version(requirement.name)
    except metadata.PackageNotFoundError:
        to_install.append(specification)
    else:
        if not requirement.specifier.contains(installed, prereleases=True):
            to_install.append(specification)
            to_update.append(requirement.name)

if to_install:
    print('Устанавливаем в окружение ядра:', sys.executable)
    print('Необходимые зависимости:', ', '.join(to_install))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *to_install])
    invalidate_caches()
    print('Зависимости установлены.')
    if to_update:
        print('Обновлены:', ', '.join(to_update))
        print('Если эти библиотеки уже были импортированы, перезапустите ядро.')
else:
    print('Все зависимости из pyproject.toml уже установлены в подходящих версиях.')


Все зависимости из pyproject.toml уже установлены в подходящих версиях.


## Загрузка данных

Скачиваем RuSpellGold, если локального файла ещё нет.


In [2]:
import shutil
import pandas as pd
from IPython.display import display
from huggingface_hub import hf_hub_download

DATA = ROOT / 'data/raw/ruspellgold/test.jsonl'
DATA.parent.mkdir(parents=True, exist_ok=True)
if not DATA.is_file() or DATA.stat().st_size == 0:
    downloaded = hf_hub_download(
        repo_id='RussianNLP/RuSpellGold',
        filename='data/complete_test/test.json',
        repo_type='dataset',
        revision='8b6694e45ac9222bc705ee4af9d6367ac083ff5c',
        local_dir=DATA.parent / '.hf',
        token=False,
    )
    temporary = DATA.with_suffix('.jsonl.tmp')
    shutil.copyfile(downloaded, temporary)
    temporary.replace(DATA)
else:
    print('RuSpellGold уже скачан.')

df = pd.read_json(DATA, lines=True)
if len(df) != 1711 or set(df.columns) != {'source', 'correction', 'domain'}:
    raise ValueError('Неожиданный формат RuSpellGold.')
print('Записей:', len(df))
pd.set_option('display.max_colwidth', 100)
display(df.head())


RuSpellGold уже скачан.
Записей: 1711


,source,correction,domain
0,"... В один из летних вечеров прозвучал тревожный звонок от знакомой – матерой путешественницы, н...","В один из летних вечеров прозвучал тревожный звонок от знакомой – матерой путешественницы, не в...",aranea
1,"В галерее еще больше промо и кадров из сериала в отличном качестве и большом размере, за что сно...","В галерее ещё больше промо и кадров из сериала в отличном качестве и большом размере, за что сно...",aranea
2,В некоторых случаях будет необходимо предоставить нотариально заверенное согласие супруга на отч...,В некоторых случаях будет необходимо предоставить нотариально заверенное согласие супруга на отч...,aranea
3,"Видела в городе афиши, анонсрующие ее концерт.","Видела в городе афиши, анонсирующие её концерт",aranea
4,Включение и выключение насоса происходит автономно с использованием отделного подключенного к не...,Включение и выключение насоса происходит автономно с использованием отдельного подключенного к н...,aranea
